# Символьный трансформер и GPT

LSTM — в `text_rnn.ipynb`. Здесь decoder-only LM на тех же текстах Kizaru.

1. **Простой трансформер** (до GPT): post-norm, ReLU в FFN, causal mask — как в Attention Is All You Need.
2. **Mini-GPT**: тот же causal LM, но pre-norm + GELU как в GPT-2.

Код в репо: `kizaru/lm.py`. Чекпоинты: `python train_lm.py --arch transformer` / `--arch gpt`.


In [ ]:
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path.cwd().resolve()
if not (ROOT / "core").is_dir() and (ROOT.parent / "core").is_dir():
    ROOT = ROOT.parent

text = (ROOT / "data" / "corpus" / "kizaru_lyrics.txt").read_text(encoding="utf-8")
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(len(text), "chars", len(chars), "vocab", device)


## Attention

Scaled dot-product: `softmax(QK^T / sqrt(d)) V`. Causal mask скрывает будущие токены: позиция `t` не видит `t+1`.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q @ k.transpose(-2, -1)) * (self.d_head ** -0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


## 1. Простой трансформер (post-norm, ReLU)


In [ ]:
class VaswaniBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.ReLU(), nn.Linear(4 * d_model, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        return self.ln2(x + self.ffn(x))


class SimpleTransformer(nn.Module):
    def __init__(self, vocab, d_model=64, n_layers=2, n_heads=4, block_size=64):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [VaswaniBlock(d_model, n_heads, block_size) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, idx):
        idx = idx[:, -self.block_size :]
        T = idx.size(1)
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


In [ ]:
def get_batch(block_size, batch_size=16):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


def train_lm(model, steps=120, lr=3e-4):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        x, y = get_batch(model.block_size)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 40 == 0:
            print(step + 1, float(loss))
    return model


@torch.no_grad()
def sample(model, start="Я", n=120, temp=0.8):
    model.eval()
    ids = [stoi[ch] for ch in start if ch in stoi] or [0]
    for _ in range(n):
        ctx = torch.tensor([ids[-model.block_size :]], device=device)
        logits = model(ctx)[0, -1] / temp
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
        ids.append(nxt)
    return "".join(itos[i] for i in ids)

tf = train_lm(SimpleTransformer(len(chars)))
print(sample(tf))


## 2. Блок GPT (pre-norm, GELU)


In [ ]:
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


class MiniGPT(nn.Module):
    def __init__(self, vocab, d_model=64, n_layers=2, n_heads=4, block_size=64):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [GPTBlock(d_model, n_heads, block_size) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, idx):
        idx = idx[:, -self.block_size :]
        T = idx.size(1)
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

gpt = train_lm(MiniGPT(len(chars)))
print(sample(gpt))


Модели в репо: `kizaru/lm.py` (`SimpleTransformer`, `MiniGPT`). Чекпоинты: `python train_lm.py --arch transformer --out checkpoints/transformer.pt` и `--arch gpt --out checkpoints/gpt.pt`. В боте: `/ai text` с `model=lstm|transformer|gpt`.
